# Experiment Plot Generator

This notebook provides a streamlined interface for generating experiment plots.

## Quick Start
1. Run cells 1-2 to load configuration and utilities
2. Use the **Interactive Mode** (Section 2) to explore individual experiments
3. Use the **Batch Mode** (Section 3) to generate all paper figures at once

In [ ]:
# Cell 1: Imports and Configuration
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import importlib

# Import our modular utilities (reload to pick up changes)
import plot_config
import plot_utils
importlib.reload(plot_config)
importlib.reload(plot_utils)

from plot_config import (
    DATASETS, get_experiment_config, COLORS, FONTSIZE,
    SELECTED_ERROR_BOUNDS, SELECTED_MEASURE_COLS,
    DEFAULT_MEASURE_COLS, DEFAULT_ERROR_BOUND,
    PLOTS_DIR,
    # Competitor config
    COMPETITORS, DEFAULT_COMPETITORS,
    get_competitor_config, get_competitor_directory,
)
from plot_utils import (
    load_experimental_data,
    plot_metric_over_queries,
    plot_metric_with_bars,  # Combined line + bar chart
    plot_total_metric_bars,
    plot_confidence_intervals,
    compute_accuracy_summary,
    generate_all_plots,
    # Per-measure analysis functions
    parse_ci_dict,
    parse_error_dict,
    get_available_measures,
    plot_relative_error_for_measure,
    plot_ci_vs_exact_for_measure,
    # Competitor comparison functions
    load_competitor_data,
    load_all_competitors_data,
    plot_competitors_comparison,
    plot_competitors_with_bars,
    plot_competitors_bars_only,
)

# Display available datasets and scenarios
print("Available Datasets & Scenarios:")
print("=" * 60)
for ds_name, ds_config in DATASETS.items():
    print(f"\n{ds_name} ({ds_config['label']})")
    print(f"  {ds_config['description']}")
    print(f"  Scenarios:")
    for sc_name, sc_config in ds_config['scenarios'].items():
        print(f"    - {sc_name}: {sc_config['dir']}")

print("\n" + "=" * 60)
print("\nAvailable Competitors:")
for comp_name, comp_config in COMPETITORS.items():
    print(f"  - {comp_name}: {comp_config['label']} ({comp_config['color']})")

print("\n" + "=" * 60)
print(f"Default params: measure_cols={DEFAULT_MEASURE_COLS}, error_bound={DEFAULT_ERROR_BOUND}")

## Section 1: Select Dataset & Scenario

Change `CURRENT_DATASET` and `CURRENT_SCENARIO` to switch between experiments. All plots below will use this selection.

In [ ]:
# Cell 2: Select the current dataset and scenario
# Change these to switch experiments - no need to modify anything else!

CURRENT_DATASET = 'synth10'       # e.g. 'taxi', 'synth10', 'synth50', 'sdss_10cols', 'sdss_100cols'
CURRENT_SCENARIO = 'pan'          # e.g. 'pan', 'pan_zoom', 'random'

# Get the configuration for the selected dataset + scenario
config = get_experiment_config(CURRENT_DATASET, CURRENT_SCENARIO)
directory = config['dir']
exp_label = config['label']
exp_measure_cols = config['default_measure_cols']  # Dataset-specific default

print(f"Selected: {config['dataset_name']} - {config['scenario_name']}")
print(f"Label: {exp_label}")
print(f"Directory: {directory}")
print(f"Default measure_cols for this dataset: {exp_measure_cols}")

# Verify data exists
try:
    df = load_experimental_data(directory)
    print(f"\nLoaded {len(df)} rows")
    print(f"Error bounds: {sorted(df['errorBound'].unique())}")
    print(f"Measure cols: {sorted(df['measure_cols'].unique())}")
except FileNotFoundError as e:
    print(f"⚠️ Error loading data: {e}")

## Section 2: Interactive Plots

Run individual cells below to generate specific plots for the selected experiment.

In [ ]:
# Plot: Response Time over Query Sequence (by Error Bound)
plot_metric_over_queries(
    directory,
    y_col='Time (sec)',
    group_by='errorBound',
    fixed_params={'measure_cols': exp_measure_cols},
    title=f'{exp_label}: Response Time by Error Bound'
)
plt.show()

In [ ]:
# Plot: Total Response Time (bar chart by Error Bound)
plot_total_metric_bars(
    directory,
    y_col='Time (sec)',
    group_by='errorBound',
    fixed_params={'measure_cols': exp_measure_cols},
    title=f'{exp_label}: Total Time by Error Bound'
)
plt.show()

In [ ]:
# Plot: Total Response Time (bar chart by Measure Columns)
plot_total_metric_bars(
    directory,
    y_col='Time (sec)',
    group_by='measure_cols',
    fixed_params={'errorBound': DEFAULT_ERROR_BOUND},
    title=f'{exp_label}: Total Time by Measure Columns'
)
plt.show()

In [ ]:
# Plot: I/O Operations over Query Sequence
plot_metric_over_queries(
    directory,
    y_col='I/Os',  # Actual column name in CSV
    group_by='errorBound',
    fixed_params={'measure_cols': exp_measure_cols},
    title=f'{exp_label}: I/O Operations by Error Bound'
)
plt.show()

In [ ]:
# Plot: Response Time with Bar Chart (combined view by Error Bound)
plot_metric_with_bars(
    directory,
    y_col='Time (sec)',
    group_by='errorBound',
    fixed_params={'measure_cols': exp_measure_cols},
    group_label='Error Bound',
    title=f'{exp_label}: Response Time by Error Bound'
)
plt.show()

In [ ]:
# Plot: Response Time with Bar Chart (combined view by Measure Columns)
plot_metric_with_bars(
    directory,
    y_col='Time (sec)',
    group_by='measure_cols',
    fixed_params={'errorBound': DEFAULT_ERROR_BOUND},
    group_label='Measure Columns',
    title=f'{exp_label}: Response Time by Measure Columns'
)
plt.show()

In [ ]:
# Plot: I/O with Bar Chart (combined view by Error Bound)
plot_metric_with_bars(
    directory,
    y_col='I/Os',
    group_by='errorBound',
    fixed_params={'measure_cols': exp_measure_cols},
    group_label='Error Bound',
    title=f'{exp_label}: I/O Operations by Error Bound'
)
plt.show()

### Per-Measure Accuracy Analysis

The following cells analyze confidence intervals and relative error for individual measures.
Use `get_available_measures(directory)` to see which measures are available in the current dataset.

In [ ]:
# Check which measures are available in the current dataset
available_measures = get_available_measures(directory)
print(f"Available measures in {CURRENT_DATASET}/{CURRENT_SCENARIO}: {available_measures}")

In [ ]:
# Plot relative error for a specific measure
# Change measure_id to analyze different measures (use available_measures from above)
measure_id = available_measures[0] if available_measures else 0

plot_relative_error_for_measure(
    directory=directory,
    measure_id=measure_id,
    fixed_params={'measure_cols': exp_measure_cols},
    include_initialization=False,
    title=f'Relative Error for Measure {measure_id} ({exp_label})',
    # save_as=f'{exp_label}_relative_error_m{measure_id}',  # Uncomment to save
    # experiment_name=exp_label,
)

In [ ]:
# Plot CI bands vs exact value for a specific measure
# Shows the estimated value with confidence interval bands for each error bound

plot_ci_vs_exact_for_measure(
    directory=directory,
    measure_id=measure_id,
    fixed_params={'measure_cols': exp_measure_cols},
    include_initialization=False,
    title=f'Confidence Intervals for Measure {measure_id} ({exp_label})',
)

In [ ]:
# Plot relative error for ALL available measures (loop)
for m_id in available_measures:
    fig = plot_relative_error_for_measure(
        directory=directory,
        measure_id=m_id,
        fixed_params={'measure_cols': exp_measure_cols},
        include_initialization=False,
        title=f'Relative Error for Measure {m_id} ({exp_label})',
    )
    plt.show()

In [ ]:
# Plot: Confidence Intervals vs Exact Results
# NOTE: This requires 'Confidence Interval LB' and 'Confidence Interval UB' columns
# which may not exist in your current data format. Check your CSV columns first.

# Uncomment if your data has the required columns:
# plot_confidence_intervals(
#     directory,
#     title=f'{exp_label}: Exact vs Approximate Results'
# )
# plt.show()

# Show available columns instead:
print("Available columns in data:")
print(df.columns.tolist())

In [ ]:
# Table: Accuracy Summary
summary = compute_accuracy_summary(directory)
print(f"\nAccuracy Summary for {exp_label}:")
print("=" * 60)
print(summary.to_string(index=False))

### Competitor Comparison

Compare Valinor against baseline methods (DuckDB CSV, DuckDB Table, etc.)
Configure which competitors to include using the `selected_competitors` list below.

In [ ]:
# Select which competitors to compare
# Available: 'valinor', 'valinor_exact', 'duckdb_csv', 'duckdb_table', 'pilotdb'
selected_competitors = DEFAULT_COMPETITORS  # from plot_config.py

# Show available competitors and their directories for the current dataset
print(f"Checking competitor data availability for {CURRENT_DATASET}/{CURRENT_SCENARIO}:")
print("=" * 60)
for comp in COMPETITORS.keys():
    comp_dir = get_competitor_directory(directory, comp)
    exists = Path(comp_dir).exists() if COMPETITORS[comp]['subdir'] else True
    status = "✓" if exists else "✗"
    print(f"  {status} {comp}: {comp_dir}")

In [ ]:
# Plot: Response Time comparison across competitors (line chart)
plot_competitors_comparison(
    base_dir=directory,
    competitors=selected_competitors,
    y_col='Time (sec)',
    measure_cols=exp_measure_cols,
    error_bound=DEFAULT_ERROR_BOUND,
    title=f'{exp_label}: Response Time Comparison',
)
plt.show()

In [ ]:
# Plot: Response Time comparison with bar chart showing totals
plot_competitors_with_bars(
    base_dir=directory,
    competitors=selected_competitors,
    y_col='Time (sec)',
    measure_cols=exp_measure_cols,
    error_bound=DEFAULT_ERROR_BOUND,
    title=f'{exp_label}: Response Time Comparison',
)
plt.show()

In [ ]:
# Plot: Total Response Time bar chart only
plot_competitors_bars_only(
    base_dir=directory,
    competitors=selected_competitors,
    y_col='Time (sec)',
    measure_cols=exp_measure_cols,
    error_bound=DEFAULT_ERROR_BOUND,
    title=f'{exp_label}: Total Response Time by Method',
)
plt.show()

### Error Bound Scaling Comparison

Compare approximate methods (Valinor vs PilotDB) across different error bounds.  
Exact methods shown as horizontal reference lines.

In [ ]:
# Error Bound Scaling: grouped bar chart comparing approximate methods
# across error bounds, with exact methods as horizontal reference lines.
from plot_config import FIGSIZE_SINGLE, FONTSIZE_SMALL, FONTSIZE_TITLE

approx_methods = ['valinor', 'pilotdb']
error_bounds = [0.01, 0.02, 0.05, 0.1]

# --- Collect average response time per method & error bound ---
scaling_data = {}  # {method: {error_bound: mean_time}}
for method in approx_methods:
    scaling_data[method] = {}
    for eb in error_bounds:
        try:
            df = load_competitor_data(
                base_dir=directory,
                competitor=method,
                measure_cols=exp_measure_cols,
                error_bound=eb,
            )
            # Exclude query 0 (initialization) and average across runs
            df_queries = df[df['i'] > 0]
            avg_time = df_queries.groupby('run')['Time (sec)'].sum().mean()
            scaling_data[method][eb] = avg_time
        except (FileNotFoundError, ValueError) as e:
            print(f"Skipping {method} error={eb}: {e}")

# --- Collect exact reference lines ---
exact_refs = {}
for method, label in [('valinor', 'VALINOR (Exact)')]:
    try:
        df_exact = load_competitor_data(
            base_dir=directory,
            competitor=method,
            measure_cols=exp_measure_cols,
            error_bound=0,
        )
        df_queries = df_exact[df_exact['i'] > 0]
        exact_refs[label] = df_queries.groupby('run')['Time (sec)'].sum().mean()
    except (FileNotFoundError, ValueError):
        pass

# Also try DuckDB baselines
for method in ['duckdb_table']:
    try:
        df_ex = load_competitor_data(
            base_dir=directory,
            competitor=method,
            measure_cols=exp_measure_cols,
        )
        df_queries = df_ex[df_ex['i'] > 0]
        exact_refs[COMPETITORS[method]['label']] = df_queries.groupby('run')['Time (sec)'].sum().mean()
    except (FileNotFoundError, ValueError):
        pass

# --- Plot ---
fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)

n_methods = len(approx_methods)
n_bounds = len(error_bounds)
bar_width = 0.8 / n_methods
x = np.arange(n_bounds)

for idx, method in enumerate(approx_methods):
    times = [scaling_data[method].get(eb, 0) for eb in error_bounds]
    offset = (idx - (n_methods - 1) / 2) * bar_width
    bars = ax.bar(
        x + offset, times, bar_width,
        label=COMPETITORS[method]['label'],
        color=COMPETITORS[method]['color'],
        edgecolor='black', linewidth=0.5,
    )
    # Add value labels on bars
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=FONTSIZE_SMALL - 2)

# Exact reference lines
ref_styles = [('--', '#d62728'), ('-.', '#ff7f0e'), (':', '#9467bd')]
for i, (label, ref_time) in enumerate(exact_refs.items()):
    style, color = ref_styles[i % len(ref_styles)]
    ax.axhline(y=ref_time, linestyle=style, color=color, linewidth=1.5,
               label=f'{label} ({ref_time:.2f}s)', alpha=0.8)

ax.set_xlabel('Error Bound (ε)', fontsize=FONTSIZE)
ax.set_ylabel('Total Response Time (s)', fontsize=FONTSIZE)
ax.set_title(f'{exp_label}: Approximate Methods vs Error Bound', fontsize=FONTSIZE_TITLE)
ax.set_xticks(x)
ax.set_xticklabels([str(eb) for eb in error_bounds], fontsize=FONTSIZE_SMALL)
ax.tick_params(axis='y', labelsize=FONTSIZE_SMALL)
ax.legend(fontsize=FONTSIZE_SMALL - 1, loc='best')
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## Section 3: Batch Generation

Generate all figures for the paper in one go. Figures are saved to the `figures/` directory.

In [ ]:
# Generate all plots for ALL experiments
# This creates paper-ready PDFs in the plots/ directory

# generate_all_plots(DATASETS)

In [ ]:
# Generate plots for a SUBSET of experiments
selected_datasets = {
    'sdss_10cols': DATASETS['sdss_10cols'],
    'taxi': DATASETS['taxi'],
}

# generate_all_plots(selected_datasets)